# DSC502 — Week 5 Session Notebook
## SQL and Structured Thinking

---

**Last week** you loaded a flat CSV with 150 QuickBite orders in 13 columns. It worked well for pandas filtering — but what happens when the table grows to 150,000 rows? What happens when a customer changes city, or a restaurant changes its name? What if you want to answer a question that spans multiple entities — not just orders, but the customers who placed them and the restaurants that fulfilled them?

That's where the **relational model** comes in.

You'll notice the data looks different this week. Instead of one big flat file, there are **five smaller tables** — customers, restaurants, drivers, orders, reviews — each storing one type of thing. SQL is the language for querying them.

---

## Two micro-lectures this week

- **Micro-lecture 1** — SQL and the relational model. Watch before Part 1.
- **Micro-lecture 2** — JOIN types reference deck. Keep it open during Part 3.


---

## Part 1 — The Relational Model and SQL Basics
### Session 1 · Copilot OFF · DuckDB CLI

> **Environment note:** Part 1 uses the **DuckDB CLI** — not a Python notebook. Open a terminal in your Codespace and follow the steps below.


### Step 1 — Launch DuckDB

In the Codespace terminal, first navigate to the week05 folder, then launch DuckDB:

```bash
cd week05
duckdb
```

You should see a `D` prompt — that's the DuckDB shell.

> **DuckDB not found?** If you get `command not found`, run:
> ```bash
> pip install duckdb --break-system-packages
> ```
> Then try `duckdb` again.

To **exit DuckDB** at any point, type `.quit` or `.exit`, or press `Ctrl+D`.


### Step 2 — Load the QuickBite schema

Once you're at the `D` prompt, load the QuickBite tables:

```sql
.read ../data/load_quickbite.sql
```

You should see a confirmation message listing the five tables. Keep this terminal open alongside the notebook.


### Step 3 — Explore the schema

Run these in the DuckDB CLI to see what tables exist and what's in them:

```sql
SHOW TABLES;
DESCRIBE orders;
DESCRIBE customers;
```

Write down the column names for `orders` and `customers`. You'll need them for the queries below.


### Step 4 — Your first SQL queries (hand-written, no Copilot)

Type each query yourself in the DuckDB CLI. Fix any errors you get — that's the point of this exercise.

**Query 1** — see the first 5 rows of the orders table:

```sql
SELECT * FROM orders LIMIT 5;
```

**Query 2** — restaurants sorted by average rating, highest first:

```sql
SELECT name, average_rating FROM restaurants ORDER BY average_rating DESC;
```

**Query 3** — the 5 highest-value orders:

```sql
SELECT order_id, total FROM orders ORDER BY total DESC LIMIT 5;
```

**Query 4** — completed orders only:

```sql
SELECT * FROM orders WHERE status = 'completed' LIMIT 10;
```

**Query 5** — orders over £40, sorted by total:

```sql
SELECT order_id, customer_id, total FROM orders WHERE total > 40 ORDER BY total DESC;
```


---

## 💬 Commentary Cell 1

*Answer in plain English below. There is no code to write here — this is a reflection.*

**Specific prompts:**

- Where did you get stuck writing SQL for the first time? What error did you get? How did you fix it?
- What is a result set? How does it feel different from a pandas DataFrame?
- Pick one query where you had to change something to get it working. What did you change and why?
- What surprised you about the schema — the column names, the data types, anything unexpected?


*Write your commentary here.*

---

## Part 2 — GROUP BY and Aggregation
### Session 2 · Copilot ON

> **Mode shift:** From here, Copilot autocomplete is **ON**. Your job is to read, run, and verify what Copilot generates — not just accept it. Keep asking: does this result make sense? Does the row count look right?


### Setting up DuckDB in the notebook

We're moving from the CLI to Python so you can combine SQL with pandas naturally. Here's what each line does:

- `duckdb.connect()` — opens an **in-memory** DuckDB database. No file is created.
- `con.execute('CREATE TABLE ... AS SELECT * FROM read_csv_auto(...)')` — reads a CSV file and creates an in-memory table from it. `read_csv_auto` detects the column types automatically.
- `con.execute('SELECT ...').df()` — runs a SQL query and returns the result as a **pandas DataFrame** you can use in Python.

Run the setup cell below, then check the output confirms five tables loaded.


In [ ]:
import duckdb

con = duckdb.connect()

# Load all five QuickBite tables from CSV
for table, path in [
    ('customers',   '../data/qb_customers.csv'),
    ('restaurants', '../data/qb_restaurants.csv'),
    ('drivers',     '../data/qb_drivers.csv'),
    ('orders',      '../data/qb_orders.csv'),
    ('reviews',     '../data/qb_reviews.csv'),
]:
    con.execute(f"""
        CREATE TABLE {table} AS
        SELECT * FROM read_csv_auto('{path}')
    """)

# Confirm tables loaded
con.execute('SHOW TABLES').df()


### Bridging from CLI to notebook — familiar SQL in a Python wrapper

The SQL you write in the CLI is exactly the same SQL you write here — it's just wrapped in `con.execute(...).df()`. Let's run the queries from Part 1 in the notebook to confirm everything looks the same.


In [ ]:
# Quick look at the orders table
con.execute('SELECT * FROM orders LIMIT 5').df()


In [ ]:
# Completed orders only
con.execute("SELECT * FROM orders WHERE status = 'completed' LIMIT 5").df()


In [ ]:
# Top 5 orders by total
con.execute('SELECT order_id, total FROM orders ORDER BY total DESC LIMIT 5').df()


In [ ]:
# Orders over £40 with AND
con.execute("""
    SELECT order_id, customer_id, total
    FROM orders
    WHERE total > 40
    AND status = 'completed'
    ORDER BY total DESC
""").df()


---

### GROUP BY — collapsing rows into groups

**GROUP BY** collapses multiple rows into one group per unique value of a column. Instead of seeing every individual row, you see one summary row per group.

The aggregate functions — `COUNT(*)`, `AVG()`, `SUM()`, `MAX()`, `MIN()` — calculate a single value for each group.


#### Worked example 1 — order count by status

Here we group all orders by their `status` column and count how many fall into each group.


In [ ]:
con.execute("""
    SELECT status,
           COUNT(*) AS order_count
    FROM orders
    GROUP BY status
    ORDER BY order_count DESC
""").df()


#### Worked example 2 — multiple aggregates by status

You can calculate several aggregates at once. Here we get the count, average, maximum, and minimum total for each status.

> **Alias tip:** `AS order_count` gives the calculated column a readable name. Without an alias, DuckDB shows an ugly auto-generated name.


In [ ]:
con.execute("""
    SELECT status,
           COUNT(*) AS order_count,
           ROUND(AVG(total), 2) AS avg_total,
           MAX(total) AS max_total,
           MIN(total) AS min_total
    FROM orders
    GROUP BY status
    ORDER BY order_count DESC
""").df()


#### Worked example 3 — HAVING (filtering after grouping)

**HAVING** filters groups *after* aggregation. Think of it as WHERE, but for groups.

- `WHERE` filters individual rows **before** grouping
- `HAVING` filters groups **after** aggregation

If you try to use `WHERE COUNT(*) > 3`, you'll get an error — because `COUNT()` doesn't exist yet at the WHERE stage. That's what HAVING is for.


In [ ]:
con.execute("""
    SELECT driver_id,
           COUNT(*) AS order_count
    FROM orders
    GROUP BY driver_id
    HAVING COUNT(*) > 3
    ORDER BY order_count DESC
""").df()


---

### Exercise 2 — GROUP BY practice (Copilot ON)

Use Copilot to generate these queries. Then run each one and check the result makes sense.

> **Prompting tip:** Copilot won't add `ORDER BY` unless you ask. Include it in your prompt deliberately — e.g. *'ordered by total revenue descending'*. This is a useful lesson about prompting specificity: the more precise your instruction, the more useful the output.

**Exercise 2.1** — Total revenue per `driver_id` from the `orders` table, highest revenue first.


In [ ]:
# Exercise 2.1 — total revenue by driver_id


**Exercise 2.2** — Average `rating` per `customer_id` from the `reviews` table.


In [ ]:
# Exercise 2.2 — average rating by customer_id


**Exercise 2.3** — Number of orders per `restaurant_id` from the `orders` table, most orders first.


In [ ]:
# Exercise 2.3 — order count by restaurant_id


**Exercise 2.4** — Number of reviews per star rating from the `reviews` table, sorted by rating descending (5 stars first).


In [ ]:
# Exercise 2.4 — review count by star rating


> **Notice something?** Your results show driver IDs and restaurant IDs — not names. To get names, you'd need to link the `orders` table to the `drivers` or `restaurants` table. That's exactly what Part 3 is about.


---

## 💬 Commentary Cell 2

*Answer in plain English below.*

**Specific prompts:**

- What does `GROUP BY` actually do? Describe it step by step: what is the input, what does SQL do to those rows, what is the output?
- What is the difference between `WHERE` and `HAVING`? When would you use each? If you got a 'WHERE on an aggregated value' error, what did the message say?
- Copilot generated a query without `ORDER BY`. What does that tell you about how to write better prompts? What did you add to get the output you actually wanted?
- Your results show IDs rather than names. What's the limitation? What would you need to make the results more readable?


*Write your commentary here.*

---

## Part 3 — JOINs and The JOIN Trap
### Session 2 · Copilot ON

> Keep **Micro-lecture 2** (the JOIN types reference deck) open while working through this part.


### What is a JOIN?

A **JOIN** combines rows from two tables based on a matching column.

In the QuickBite schema, `orders` stores a `customer_id` — but not the customer's name. The customer's name is in the `customers` table. To get the name alongside the order, you JOIN the two tables on the column they share: `customer_id`.

The `ON` clause specifies which columns to match:

```sql
FROM orders
INNER JOIN customers ON orders.customer_id = customers.customer_id
```

This tells SQL: *for each row in orders, find the row in customers where the customer_id matches, and attach it.*


### Worked example — INNER JOIN

An INNER JOIN returns only rows where a matching value exists **in both tables**. If a customer has no orders, they won't appear. If an order has no matching customer, it won't appear either.


In [ ]:
con.execute("""
    SELECT customers.name, orders.total, orders.status
    FROM orders
    INNER JOIN customers ON orders.customer_id = customers.customer_id
    LIMIT 10
""").df()


Only rows where `orders.customer_id` matches `customers.customer_id` appear. Customers with no orders, and orders with no matching customer, are both excluded.


### Worked example — LEFT JOIN

A **LEFT JOIN** keeps **all rows from the left table** (the first one listed after `FROM`), even if there is no match in the right table. Where there's no match, the right table's columns are filled with `NULL`.

This is useful when you want to see customers who *haven't* placed any orders — they'll appear with `NULL` in the order columns.


In [ ]:
con.execute("""
    SELECT customers.name, orders.total, orders.status
    FROM customers
    LEFT JOIN orders ON customers.customer_id = orders.customer_id
""").df()


Scroll to the bottom. Customers with no orders appear with `NULL` in `total` and `status`. That's the LEFT JOIN doing its job — keeping all left-side rows.


### NULLs in a LEFT JOIN result

When a LEFT JOIN finds no match, it doesn't drop the row — it fills the right-side columns with `NULL`. You can use this deliberately:

```sql
WHERE orders.order_id IS NULL
```

...finds customers who have *never* placed an order. This is a common pattern.


---

### The JOIN Trap

Here's a real problem that catches people out — and produces wrong results with no error message.

**Suppose you want total revenue per restaurant.** You write a query joining `restaurants` to `orders`. That gives you the right answer. Then Copilot (or you) adds a JOIN to `reviews` as well — to get review data — without thinking about what that does to the row count.

**What actually happens:** each order gets one row per review. Orders with no reviews disappear entirely (because INNER JOIN only keeps matching rows). Orders with one review appear once. Orders with two reviews appear twice. The revenue calculation uses only the orders that have reviews — silently excluding the rest, and giving you a wrong total with no warning.

**The trap is exclusion, not inflation.** The result is *smaller* and *wrong* — not because the numbers are inflated, but because whole orders are silently dropped.

**How to catch it:** always check the row count before and after adding a JOIN. If the count changes in a way you can't explain, the JOIN is doing something unintended.


#### Query A — correct: total revenue per restaurant from orders


In [ ]:
# Query A — total revenue per restaurant (correct)
result_a = con.execute("""
    SELECT restaurants.name,
           SUM(orders.total) AS total_revenue_all_orders
    FROM restaurants
    JOIN orders ON restaurants.restaurant_id = orders.restaurant_id
    GROUP BY restaurants.name
    ORDER BY total_revenue_all_orders DESC
""").df()

# Check row count before grouping — this is how you verify the JOIN is right
count_a = con.execute("""
    SELECT COUNT(*) AS row_count
    FROM restaurants
    JOIN orders ON restaurants.restaurant_id = orders.restaurant_id
""").df()

print('Rows before GROUP BY:', count_a['row_count'].values[0])
result_a


Query A is correct. The row count before grouping equals the number of orders — every order is accounted for.


#### Flow: now add a JOIN to reviews — watch what happens

Run the row count check first, then look at the revenue figure. Notice that some restaurants disappear entirely.


In [ ]:
# Check row count when we add a JOIN to reviews
count_b = con.execute("""
    SELECT COUNT(*) AS row_count
    FROM restaurants
    JOIN orders ON restaurants.restaurant_id = orders.restaurant_id
    JOIN reviews ON orders.order_id = reviews.order_id
""").df()

print('Rows with reviews JOIN:', count_b['row_count'].values[0])
print('Rows without reviews JOIN:', count_a['row_count'].values[0])
print('Difference:', count_a['row_count'].values[0] - count_b['row_count'].values[0], 'orders silently excluded')


In [ ]:
# Query B — same question but now joined to reviews — watch the revenue change
result_b = con.execute("""
    SELECT restaurants.name,
           SUM(orders.total) AS total_revenue_reviewed_orders_only
    FROM restaurants
    JOIN orders ON restaurants.restaurant_id = orders.restaurant_id
    JOIN reviews ON orders.order_id = reviews.order_id
    GROUP BY restaurants.name
    ORDER BY total_revenue_reviewed_orders_only DESC
""").df()

result_b


Some restaurants have disappeared — those with no reviews at all are excluded by the INNER JOIN. The remaining revenue figures are lower because only orders with a review are counted.


In [ ]:
# Compare the two results side by side
import pandas as pd
comparison = result_a.merge(result_b, on='name', how='left')
comparison


**What happened?** Restaurants with no reviews at all don't appear in Query B. For the ones that do appear, the revenue is lower because only orders *with a review* are included. No error — just a silently wrong answer.

**The fix:** only JOIN to a table if you actually need columns from it for your SELECT. If you're calculating order revenue, you don't need to join to reviews at all.


---

### Exercise 3 — JOIN practice (Copilot ON)

> **Standing instruction:** for every query in this exercise, **check the row count before accepting the result**. Add a COUNT(*) query before each main query. If the count looks wrong, the JOIN is doing something unexpected.


**Exercise 3.1** — Customer names and their order totals (INNER JOIN).

Use Copilot to generate this. Check the row count first.


In [ ]:
# Exercise 3.1 — row count check first


In [ ]:
# Exercise 3.1 — customer names with order totals


#### Table aliases

Copilot may or may not use table aliases. An alias is a shorthand name for a table:

```sql
FROM orders AS o
INNER JOIN customers AS c ON o.customer_id = c.customer_id
```

`o.customer_id` means the same as `orders.customer_id` — it's just shorter. If Copilot didn't use aliases, try rewriting the query with them. You'll see both styles in real SQL — get comfortable reading either.


**Exercise 3.2** — Completed orders with customer name and restaurant name (INNER JOIN across three tables). Check row count first.

> **Hint:** Think carefully about where the `status = 'completed'` filter goes. When using LEFT JOIN, placing a filter in `WHERE` vs `ON` gives different results — in `WHERE` it silently removes NULL rows, which defeats the purpose of a LEFT JOIN. With INNER JOIN here it doesn't matter, but get into the habit of thinking about it.


In [ ]:
# Exercise 3.2 — row count check


In [ ]:
# Exercise 3.2 — completed orders with customer and restaurant name


**Exercise 3.3** — Customers who have never placed an order (LEFT JOIN + IS NULL).

Check the row count first — you should see some customers with NULL order data.


In [ ]:
# Exercise 3.3 — row count check


In [ ]:
# Exercise 3.3 — customers with no orders
con.execute("""
    SELECT c.*
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
""").df()


**Exercise 3.4** — Drivers who have no assigned orders (LEFT JOIN + IS NULL).


In [ ]:
# Exercise 3.4 — drivers with no orders


---

## 💬 Commentary Cell 3

*Answer in plain English below.*

**Specific prompts:**

- What is the difference between INNER JOIN and LEFT JOIN? Give an example from the QuickBite data where each would give a different result.
- What is the JOIN trap? Describe it in your own words — what goes wrong, why there's no error message, and how you catch it.
- You checked row counts before and after adding a JOIN. Did the count change in a way you didn't expect? What did that tell you?
- One thing you'd do differently next time you write a query with multiple JOINs.


*Write your commentary here.*

---

## End of Week 5 Session

Save your work and push to GitHub:

```bash
git add . && git commit -m 'week 5 session complete' && git push
```

Post your repo URL in Blackboard if you haven't already.
